# t0 und Timestral: neue Foundation Models, nur live

Dieses Notebook lädt die beiden neuen Foundation Models **t0-gamma** (The Forecasting Company) und **Timestral-209M** (Mistral AI), prüft sie und bereitet sie für die App vor.

**Warum nur live?** Beide sind im September bzw. Oktober 2026 erschienen, also **nach** train, val und test (bis August 2026). Einen Trainings-Cutoff nennen sie nicht. Sie könnten unsere Perioden also im Pretraining gesehen haben (**Kontamination**). Deshalb:

- In der App laufen sie **separat** mit: als eigene Modelle und in einer **zweiten Kombination**. Die Hauptprognose (Chronos-2, TimesFM-2.5, HAR, GARCH mit statischen Gewichten) und die Kursspanne bleiben unverändert.
- **Sauber ausgewertet** werden sie nur über das Logbuch der App: Jede Prognose wird gespeichert, bevor der Zieltag eintritt.
- Die **Historie** (Abschnitt 5) ist optional. Sie ist nur für einen ersten Eindruck und später für Features gedacht und gilt als *möglicherweise kontaminiert*. Daraus werden keine Gewichte gelernt.

**Setup wie bei Chronos und TimesFM:** log-RV als Input, Kontext 1024 Tage, Horizont 5 (Schritte 1 und 5), 21 Quantile, Mittelwert durch Integration der Quantile.

### Installation (einmalig, in der `.venv`)

```
pip install "tfc-t0[torch]>=0.6.0"
pip install git+https://github.com/mistralai/timestral.git
```

Beim ersten Laden werden die Gewichte von Hugging Face geladen (t0-gamma ca. 1 GB, Timestral-209M ca. 840 MB). Danach liegen sie im Cache, und die App findet sie dort.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

DATA_DIR, RESULTS_DIR, LIVE_DIR = Path("data"), Path("results"), Path("data") / "live"

# ---- Setup wie in Notebook 1 und 02_timesfm.ipynb ----
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
HORIZONS = [1, 5]
QUANTILES = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
             0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
QCOLS = [f"q{q:.2f}" for q in QUANTILES]
MIN_HISTORY = 500
CONTEXT = 1024

# ---- Neue Modelle ----
T0_REPO = "theforecastingcompany/t0-gamma"
TIMESTRAL_REPO = "mistralai/Timestral-209M"
BATCH = 64                         # Kontexte pro Aufruf bei der Historie

# ---- Historie (Abschnitt 5) ----
RUN_HISTORY = False                # True: ganze Historie rechnen (dauert, siehe Laufzeit in Abschnitt 3)

_LV = np.asarray(QUANTILES)
_QMASS = (np.r_[_LV[1:], 1.0] - np.r_[0.0, _LV[:-1]]) / 2     # Wahrscheinlichkeitsmasse je Quantil (Trapez)
_QMASS = _QMASS / _QMASS.sum()

print("torch", torch.__version__, "| threads:", torch.get_num_threads())

## 1. Daten

Wie in `02_timesfm.ipynb`. Für den Live-Check (Abschnitt 4) wird zusätzlich der aktuelle Stand der App aus `data/live/` gelesen, falls vorhanden.

In [ ]:
class RVSeries:
    def __init__(self, sid, df):
        self.sid = sid
        self.dates = df.index
        self.rv = df["rv"].to_numpy(float)
        self.logrv = np.log(self.rv)
        self.origins = np.arange(MIN_HISTORY - 1, len(df) - max(HORIZONS))


def contexts_for(s, origins):
    L = s.logrv.astype(np.float32)
    return [L[max(0, t - CONTEXT + 1): t + 1] for t in origins]


def mean_from_quantiles(q):
    # wie in Notebook 1: E[Y] als Integral der Quantilfunktion
    return q @ _QMASS


def load_daily(sym):
    live = LIVE_DIR / f"{sym}_daily.parquet"
    return pd.read_parquet(live if live.exists() else DATA_DIR / f"{sym}_daily.parquet")


series = {sym: RVSeries(sym, pd.read_parquet(DATA_DIR / f"{sym}_daily.parquet")) for sym in SYMBOLS}
for s in series.values():
    print(f"{s.sid}: {len(s.origins)} Ursprünge, Daten bis {s.dates[-1]:%d.%m.%Y}")

## 2. Die beiden Experten

Beide liefern direkt Quantile. Anders als bei TimesFM ist keine Ergänzung der Ränder nötig:

- **t0-gamma:** Man übergibt die gewünschten Quantile. t0-gamma ist auf genau unseren 21 Stufen trainiert.
- **Timestral-209M:** liefert immer seine 21 trainierten Stufen. Die Klasse prüft, dass sie mit `QUANTILES` übereinstimmen. Andernfalls wird in Logs auf der Normal-Skala interpoliert (wie bei TimesFM), und es kommt eine Warnung.

**Einstellungen:**

| | t0-gamma | Timestral-209M |
|---|---|---|
| Kontexte | jeder Ursprung für sich, Batch nur zur Beschleunigung | ebenso |
| `symmetric_averaging` | True (vom Hersteller empfohlen) | True (Default) |
| `trim_leading_missing` | True (Kontexte am Anfang der Reihe sind kürzer) | – (kürzere Kontexte werden mit NaN aufgefüllt = fehlend) |
| `clip_nonnegative` | False | False |

`clip_nonnegative` ist ausgeschaltet, weil die log-RV negativ ist. Kürzere Kontexte (die ersten Ursprünge der Historie) werden links mit NaN aufgefüllt. Beide Modelle behandeln NaN als fehlenden Wert.

Danach dieselbe Rücktransformation wie bei Chronos und TimesFM: `exp`, sortieren, Mittelwert durch Integration der Quantile.

In [ ]:
def pad_left(ctx):
    # Liste von Kontexten unterschiedlicher Länge -> (B, T) mit NaN links
    T = max(len(c) for c in ctx)
    out = np.full((len(ctx), T), np.nan, dtype=np.float32)
    for i, c in enumerate(ctx):
        out[i, T - len(c):] = c
    return out


def to_grid(levels, logq):
    # (..., Q) Log-Quantile auf `levels` -> (..., 21) auf QUANTILES; exakt, wenn die Stufen gleich sind
    levels = np.asarray(levels, float)
    if len(levels) == len(_LV) and np.allclose(levels, _LV):
        return logq
    from scipy.stats import norm
    z_src, z_dst = norm.ppf(levels), norm.ppf(_LV)
    flat = logq.reshape(-1, len(levels))
    out = np.empty((len(flat), len(_LV)))
    for i, row in enumerate(flat):
        lo = (row[1] - row[0]) / (z_src[1] - z_src[0])
        hi = (row[-1] - row[-2]) / (z_src[-1] - z_src[-2])
        out[i] = np.interp(z_dst, z_src, row)
        out[i] = np.where(z_dst < z_src[0], row[0] + lo * (z_dst - z_src[0]), out[i])
        out[i] = np.where(z_dst > z_src[-1], row[-1] + hi * (z_dst - z_src[-1]), out[i])
    return out.reshape(logq.shape[:-1] + (len(_LV),))


def finish(log_q):
    # (n, H, 21) Log-Quantile -> mean (n, nH), q (n, nH, 21) für HORIZONS
    log_q = np.asarray(log_q, float)[:, [h - 1 for h in HORIZONS], :]
    q = np.sort(np.exp(log_q), axis=-1)
    return mean_from_quantiles(q), q


class T0Gamma:
    name = "t0-gamma"

    def __init__(self, repo=T0_REPO, batch=BATCH):
        from t0 import T0Forecaster
        self.batch = batch
        self.model = T0Forecaster.from_pretrained(repo).eval()

    def predict_log(self, ctx):
        out = self.model.predict(pad_left(ctx), horizon=max(HORIZONS), quantile_levels=QUANTILES,
                                 trim_leading_missing=True, symmetric_averaging=True,
                                 clip_nonnegative=False, snap_integers=False)
        q = out.quantiles
        q = q.detach().float().cpu().numpy() if hasattr(q, "detach") else np.asarray(q, float)
        return q.reshape(len(ctx), max(HORIZONS), len(QUANTILES))

    def forecast(self, s, origins):
        ctx = contexts_for(s, origins)
        parts = [self.predict_log(ctx[k: k + self.batch]) for k in range(0, len(ctx), self.batch)]
        return finish(np.concatenate(parts))


class Timestral:
    name = "Timestral-209M"

    def __init__(self, repo=TIMESTRAL_REPO, batch=BATCH):
        from timestral import TimestralForecaster
        self.batch = batch
        self.model = TimestralForecaster.from_pretrained(repo, device="cuda" if torch.cuda.is_available() else "cpu")
        self.levels = tuple(self.model.quantile_levels)
        same = len(self.levels) == len(_LV) and np.allclose(self.levels, _LV)
        if not same:
            print("Achtung: Timestral liefert andere Quantilstufen, es wird interpoliert:", self.levels)

    def predict_log(self, ctx):
        out = self.model.forecast(pad_left(ctx)[:, None, :], prediction_length=max(HORIZONS),
                                  symmetric_averaging=True, clip_nonnegative=False)
        q = out.quantiles.detach().float().cpu().numpy()[:, 0]          # (B, H, Q)
        return to_grid(self.levels, q)

    def forecast(self, s, origins):
        ctx = contexts_for(s, origins)
        parts = [self.predict_log(ctx[k: k + self.batch]) for k in range(0, len(ctx), self.batch)]
        return finish(np.concatenate(parts))

## 3. Laden, Plausibilität und Laufzeit

Beide Modelle werden geladen und auf den letzten 64 Ursprüngen von BTC gerechnet. Zum Vergleich stehen die gespeicherten Prognosen von Chronos-2 (voll) und TimesFM-2.5 für dieselben Tage daneben. Die Werte sollten in derselben Grössenordnung liegen. Aus der Laufzeit wird hochgerechnet, wie lange die ganze Historie dauern würde.

In [ ]:
experts = {}
for cls in [T0Gamma, Timestral]:
    t = time.time()
    try:
        experts[cls.name] = cls()
        print(f"{cls.name}: geladen in {time.time() - t:.0f} s")
    except Exception as e:
        print(f"{cls.name}: konnte nicht geladen werden -> {type(e).__name__}: {e}")

In [ ]:
s = series["BTCUSDT"]
probe = s.origins[-64:]
ann = lambda x: np.sqrt(365 * np.asarray(x)) * 100       # Tagesvarianz -> annualisierte Vola in %

check = {}
for name, ex in experts.items():
    t = time.time()
    mean, q = ex.forecast(s, probe)
    sec = (time.time() - t) / len(probe)
    n_all = sum(len(x.origins) for x in series.values())
    print(f"{name}: {1000 * sec:.0f} ms pro Ursprung -> ganze Historie ca. {sec * n_all / 60:.0f} Min.")
    assert np.isfinite(q).all() and (np.diff(q, axis=-1) >= 0).all(), "Quantile nicht endlich oder nicht sortiert"
    check[name] = mean

for m in ["Chronos-2-full", "TimesFM-2.5"]:
    p = RESULTS_DIR / f"BTCUSDT__{m}.parquet"
    if p.exists():
        d = pd.read_parquet(p).set_index(["origin", "h"])
        check[m] = np.stack([d.loc[[(s.dates[t], h) for t in probe], "mean"].to_numpy() for h in HORIZONS], axis=1)

truth = np.stack([s.rv[probe + h] for h in HORIZONS], axis=1)
rows = {m: {f"Vola h={h} (Mittel, %)": ann(v[:, j]).mean().round(1) for j, h in enumerate(HORIZONS)}
        for m, v in {**check, "tatsächlich": truth}.items()}
display(pd.DataFrame(rows).T)

## 4. Live-Check: dieselbe Prognose wie in der App

Die App rechnet mit dem letzten bekannten Tag als Ursprung. Hier wird genau das für alle Coins gemacht, mit dem aktuellen Datenstand der App (`data/live/`). Danach in der App auf «Daten laden und Prognose berechnen» klicken: Die Werte für t0 und Timestral im Reiter des Coins sollten dieselben sein.

In [ ]:
rows = []
for sym in SYMBOLS:
    d = load_daily(sym)
    ctx = [np.log(d["rv"].to_numpy(float)).astype(np.float32)[-CONTEXT:]]
    for name, ex in experts.items():
        mean, q = finish(ex.predict_log(ctx))
        for j, h in enumerate(HORIZONS):
            rows.append({"Coin": sym, "Datenstand": f"{d.index[-1]:%d.%m.%Y}", "Modell": name, "h": h,
                         "Erwartet (%)": ann(mean[0, j]).round(1),
                         "90 % von (%)": ann(q[0, j, 1]).round(1), "90 % bis (%)": ann(q[0, j, 19]).round(1)})
display(pd.DataFrame(rows).set_index(["Coin", "Datenstand", "Modell", "h"]))

## 5. Historie (optional, möglicherweise kontaminiert)

Nur mit `RUN_HISTORY = True`. Schreibt `results/<COIN>__t0-gamma.parquet` und `results/<COIN>__Timestral-209M.parquet` im selben Format wie die anderen Experten. **Bestehende Dateien werden nicht überschrieben.**

Diese Dateien sind **nicht** Teil der Hauptauswertung: `01_kombination.ipynb` und die App verwenden sie nicht, solange die Modelle nicht in `POOL` stehen. Die Spalte `split` wird absichtlich auf `"kontaminiert?"` gesetzt, damit niemand sie versehentlich als saubere Evaluation liest.

In [ ]:
def run_history(expert, s):
    path = RESULTS_DIR / f"{s.sid}__{expert.name}.parquet"
    if path.exists():
        print(f"{path.name} existiert schon, wird nur geladen")
        return pd.read_parquet(path)
    t = time.time()
    mean, q = expert.forecast(s, s.origins)
    runtime = time.time() - t
    parts = []
    for j, h in enumerate(HORIZONS):
        tgt = s.origins + h
        df = pd.DataFrame({"sid": s.sid, "model": expert.name, "h": h,
                           "origin": s.dates[s.origins], "target_date": s.dates[tgt],
                           "y": s.rv[tgt], "mean": mean[:, j]})
        parts.append(pd.concat([df, pd.DataFrame(q[:, j, :], columns=QCOLS)], axis=1))
    out = pd.concat(parts, ignore_index=True)
    out["split"] = "kontaminiert?"
    out.to_parquet(path)
    with open(RESULTS_DIR / "runtimes_new_models.csv", "a") as f:
        f.write(f"{s.sid},{expert.name},{runtime:.1f}\n")
    print(f"{expert.name:>16} | {s.sid}: {runtime:.0f}s -> {path.name}")
    return out


if RUN_HISTORY:
    for ex in experts.values():
        for s in series.values():
            run_history(ex, s)
else:
    print("RUN_HISTORY = False: Historie wird nicht gerechnet.")

## 6. Wie es in der App weitergeht

**Separat angezeigt:** Im Reiter jedes Coins stehen t0 und Timestral unter «Was die einzelnen Modelle sagen» in einem eigenen Block «neu, nur live», zusammen mit der **Kombination mit neuen Modellen**.

**Startgewichte der zweiten Kombination:** Die beiden Foundation Models machen fast dieselben Fehler (Fehlerkorrelation 0.98). Deshalb wird ihr bisheriges Gesamtgewicht aus den statischen Gewichten gleichmässig auf alle vier Foundation Models verteilt. HAR und GARCH behalten ihr Gewicht:

| | h=1 | h=5 |
|---|---|---|
| Chronos-2, TimesFM-2.5, t0, Timestral | je ¼ von 0.93 ≈ 0.23 | je ¼ von 0.77 ≈ 0.19 |
| HAR | 0.00 | 0.03 |
| GARCH | 0.07 | 0.20 |

Die App rechnet diese Werte aus den aktuellen statischen Gewichten, die Tabelle ist gerundet. Fehlt ein Foundation Model, wird sein Anteil auf die übrigen verteilt.

**Auswertung:** Das Logbuch zeigt QLIKE und Trefferquote auch für t0, Timestral und die zweite Kombination. Ihre Zahl an Prognosen ist kleiner, weil sie erst ab jetzt mitlaufen. Erst nach einigen Monaten lässt sich sagen, ob die zweite Kombination besser ist als die Hauptprognose.